In [1]:
%pip install -q --upgrade pymcel rebound montu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Aberración de la luz en estrellas

In [51]:
import numpy as np
import matplotlib.pyplot as plt
import montu as mn
import rebound as rb
import pymcel as pc

In [52]:
tabla, jd, X = pc.consulta_horizons(id='399',
                                    location='@0',
                                    epochs ='2025-03-24 15:40:00'
                )


In [53]:
v_tierra = X[3:]
v_tierra

array([ 1.54361812e+03, -2.98426205e+04,  8.27267631e-01])

Ahora el beta de la estrella

In [54]:
beta_vec = -v_tierra / pc.constantes.c
beta = np.linalg.norm(beta_vec)
beta_vec, beta 

(array([-5.14895583e-06,  9.95442672e-05, -2.75946779e-09]),
 np.float64(9.9677343879579e-05))

In [55]:
gamma = 1/np.sqrt(1-beta**2)
gamma

np.float64(1.0000000049677866)

In [56]:
allstars = mn.Stars()

Loading stellar catalogue montu_stellar_catalogue_v38.csv


In [57]:
star = allstars.get_stars(ProperName='Aldebaran')
star

1 star(s):
|    |   MN |    HD |   HR |   HIP | Gl        | Name      | OtherDesignations                                                                     | ProperName   | Bayer   | Flamsteed   | Constellation   |   RAJ2000 |   DecJ2000 |   GalLonJ2000 |   GalLatJ2000 |   pmRA |   pmDec |   RadVel |   Distance |   Vmag |   Vmag_min |   Vmag_max |   B-V | SpType   |   Luminosity |   XJ2000 |   YJ2000 |   ZJ2000 |   VXJ2000 |   VYJ2000 |   VZJ2000 |   Primary | MultipleID   |   IsMultiple |   IsVariable |
|----|------|-------|------|-------|-----------|-----------|---------------------------------------------------------------------------------------|--------------|---------|-------------|-----------------|-----------|------------|---------------|---------------|--------|---------|----------|------------|--------|------------|------------|-------|----------|--------------|----------|----------|----------|-----------|-----------|-----------|-----------|--------------|--------------|---

In [58]:
ra = np.array(star.data.RAJ2000)[0]
dec = np.array(star.data.DecJ2000)[0]
ra, dec

(np.float64(4.598677), np.float64(16.509301))

In [59]:
import spiceypy as spy
deg = np.pi/180
rad = 1 / deg

In [60]:
nprima_equ = spy.latrec(1, ra*15*deg, dec*deg)
nprima_equ

array([0.34390374, 0.89497322, 0.28417099])

### Transformar al sistema eclíptico de coordenadas

In [61]:
Requ2ecl = spy.pxform('J2000', 'ECLIPJ2000', 0) #Matriz de rotación
Requ2ecl

array([[ 1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.91748206,  0.39777716],
       [ 0.        , -0.39777716,  0.91748206]])

Ya se puede convertir el vector

In [62]:
nprima_ecl = spy.mxv(Requ2ecl, nprima_equ)
nprima_ecl
n = (nprima_equ + (((gamma -1)/beta**2)*np.dot(beta_vec, nprima_equ) + gamma)*beta_vec) / (gamma*(1+np.dot(beta_vec, nprima_equ)))

### Quiero calcular la ascención recta y la declinación después de la aberración

In [63]:
n_equ = spy.mxv(spy.invert(Requ2ecl),n)
n_equ

array([0.34386856, 0.70811465, 0.61670743])

In [64]:
r, long, lat = spy.reclat(n_equ)

In [70]:
ra_aberrada = long * rad / 15
dec_aberrada = lat * rad
mn.Util.dec2hex(ra_aberrada), mn.Util.dec2hex(dec_aberrada)

('04:16:23.583', '38:04:33.925')